# Veracust — ODSS API/streaming smoke test

Purpose: verify that the ODSS Hugging Face dataset can be accessed as a stream, inspect its real schema, and decode a few audio examples in Colab. This notebook does **not** download a full archive, stage a training corpus, or start training.

Run top-to-bottom on a fresh Google Colab runtime. Audio examples are fetched on demand and held only in notebook memory for this probe.

In [ ]:
# Install only the dataset/audio dependencies. Do not reinstall PyTorch here.
%pip -q install -U "datasets[audio]" soundfile


In [ ]:
import json
from itertools import islice
from datasets import load_dataset, get_dataset_config_names, get_dataset_split_names

DATASET_ID = "SpeechAntiSpoofingBenchmarks/ODSS"
MAX_ROWS_TO_INSPECT = 5

print("Dataset:", DATASET_ID)
try:
    configs = get_dataset_config_names(DATASET_ID)
    print("Configs:", configs)
except Exception as exc:
    configs = []
    print("Could not enumerate configs:", repr(exc))

try:
    split_names = get_dataset_split_names(DATASET_ID)
    print("Splits:", split_names)
except Exception as exc:
    split_names = []
    print("Could not enumerate splits:", repr(exc))

# Pick a split only after discovering what the Hub currently exposes.
PREFERRED_SPLIT = "test"
if split_names:
    SPLIT = PREFERRED_SPLIT if PREFERRED_SPLIT in split_names else split_names[0]
else:
    SPLIT = PREFERRED_SPLIT  # If this fails, set it to a split shown on the dataset card.
print("Probe split:", SPLIT)


In [ ]:
# Open a streaming iterator. No complete dataset archive is downloaded.
stream = load_dataset(DATASET_ID, split=SPLIT, streaming=True)
print("Feature schema:")
print(stream.features)

rows = list(islice(iter(stream), MAX_ROWS_TO_INSPECT))
print(f"\nRows fetched for inspection: {len(rows)}")
if not rows:
    raise RuntimeError("The selected split returned no rows. Check the dataset card and split name.")

for i, row in enumerate(rows):
    print(f"\n--- ROW {i} ---")
    print("keys:", sorted(row.keys()))
    for key, value in row.items():
        if key == "audio":
            if isinstance(value, dict):
                print("audio metadata:", {k: v for k, v in value.items() if k != "array"})
                arr = value.get("array")
                if arr is not None:
                    print("decoded array shape:", getattr(arr, "shape", None))
            else:
                print("audio value type:", type(value).__name__)
        else:
            print(f"{key}:", repr(value)[:1200])


In [ ]:
# Inspect the documented notes field without assuming its exact live structure.
def parse_notes(value):
    if isinstance(value, dict):
        return value
    if isinstance(value, str):
        try:
            parsed = json.loads(value)
            return parsed if isinstance(parsed, dict) else {"_parsed_value": parsed}
        except json.JSONDecodeError:
            return {"_raw_notes": value}
    return {}

for i, row in enumerate(rows):
    print(f"row {i} notes:", parse_notes(row.get("notes")))
    print(f"row {i} candidate label fields:", {k: row.get(k) for k in ("label", "labels", "class", "target") if k in row})


In [ ]:
# Decode probe: verify that a few streamed examples expose actual audio samples.
# This cell deliberately does not write audio files or create a training manifest.
import numpy as np

decoded = 0
for i, row in enumerate(rows):
    audio = row.get("audio")
    if not isinstance(audio, dict) or audio.get("array") is None:
        print(f"row {i}: no decoded audio array in the 'audio' field")
        continue
    samples = np.asarray(audio["array"])
    sr = audio.get("sampling_rate")
    if samples.size == 0 or not sr:
        print(f"row {i}: invalid/empty audio payload (shape={samples.shape}, sr={sr})")
        continue
    duration = samples.shape[-1] / float(sr)
    print(f"row {i}: OK | shape={samples.shape} | sample_rate={sr} Hz | duration={duration:.2f}s | dtype={samples.dtype}")
    decoded += 1

print(f"\nDecoded audio rows: {decoded}/{len(rows)}")
if decoded == 0:
    raise RuntimeError("No audio rows decoded. Inspect the schema/output above before proceeding.")


## What to report back

After running the notebook, share the output for: **Configs**, **Splits**, **Feature schema**, one representative row's keys/metadata (omit any large audio arrays), and the **Decoded audio rows** count.

Do not start full training yet. We will use the observed schema—not guesses—to implement the English filter, label mapping, speaker/generator mapping, bounded temporary cache, and manifest validator. If the dataset card or runtime reports a licence change, stop and verify it before commercial use.